# TurboQuant Search — Quick Start

**Vector compression for similarity search** — 6-10x compression, 84-92% Recall@10, zero training.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tarun-ks/turboquant_search/blob/main/notebooks/quickstart.ipynb)
[![PyPI](https://img.shields.io/pypi/v/turboquant-search)](https://pypi.org/project/turboquant-search/)

## 1. Install

In [ ]:
!pip install -q turboquant-search[all]

## 2. Quick Demo — Index 10K Vectors, Search, Show Stats

In [ ]:
from turboquant_search import TurboQuantSearchIndex
import numpy as np

# Simulate 10K document embeddings of dimension 128
np.random.seed(42)
document_embeddings = np.random.randn(10000, 128).astype(np.float32)

# Create a compressed index — no training needed
index = TurboQuantSearchIndex(dim=128, bits=3)
index.add(document_embeddings)

# Search with a query embedding
query = np.random.randn(1, 128).astype(np.float32)
scores, top_k = index.search(query, k=10)

print(f"Top 10 results: {top_k[0]}")
print(f"Scores: {scores[0].round(4)}")

# Show compression stats
stats = index.stats()
for k, v in stats.items():
    print(f"  {k}: {v}")

## 3. Real Data Demo — Wikipedia Embeddings

Download pre-embedded Wikipedia paragraphs and build a searchable index.

In [ ]:
from turboquant_search.dataset_hub import load_dataset

# Download and cache Wikipedia embeddings (100K paragraphs, 384-dim)
vectors, queries, texts, info = load_dataset("wikipedia", n_vectors=50000)
print(f"Dataset: {info['description']}")
print(f"Loaded {info['count']:,} vectors, dim={info['dim']}")

# Build index
wiki_index = TurboQuantSearchIndex(dim=info['dim'], bits=3)
wiki_index.add(vectors)

# Search with a sample query
scores, top_k = wiki_index.search(queries[:1], k=5)
print(f"\nTop 5 results for sample query:")
for rank, (idx, score) in enumerate(zip(top_k[0], scores[0])):
    print(f"  {rank+1}. [{idx}] score={score:.4f} — {texts[idx][:100]}")

## 4. Side-by-Side Comparison — TurboQuant vs FAISS

Run both methods on the same data and compare recall.

In [ ]:
from turboquant_search import run_benchmark
from turboquant_search.benchmarks import format_results_table

# Run full benchmark on synthetic data
results = run_benchmark(
    dataset_name="synthetic",
    n_vectors=10000,
    n_queries=200,
    k_values=[1, 5, 10, 50],
    bit_widths=[2, 3, 4],
)

print(format_results_table(results))

In [ ]:
# Visualize the results
import matplotlib.pyplot as plt
import numpy as np

methods = results["methods"]
names = list(methods.keys())
recall_10 = [methods[n]["recall"][10] * 100 for n in names]
memory_mb = [methods[n]["memory_mb"] for n in names]

short_names = []
for n in names:
    if "Flat" in n: short_names.append("Flat")
    elif "IVF" in n: short_names.append("IVF-PQ")
    elif "PQ" in n and "TurboQuant" not in n: short_names.append("FAISS PQ")
    elif "TurboQuant" in n:
        bits = n.split("-bit")[0].split()[-1]
        short_names.append(f"TQ {bits}b")
    else: short_names.append(n[:8])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ["#94a3b8" if "Flat" in n else "#fb923c" if "PQ" in n and "TurboQuant" not in n
          else "#f87171" if "IVF" in n else "#3b82f6" if "4" in n
          else "#8b5cf6" if "3" in n else "#ec4899"
          for n in names]

ax1.bar(short_names, recall_10, color=colors)
ax1.set_ylabel("Recall@10 (%)")
ax1.set_title("Recall@10")
ax1.set_ylim(0, 110)
for i, v in enumerate(recall_10):
    ax1.text(i, v + 2, f"{v:.0f}%", ha="center", fontweight="bold")

ax2.bar(short_names, memory_mb, color=colors)
ax2.set_ylabel("Memory (MB)")
ax2.set_title("Memory Usage")
for i, v in enumerate(memory_mb):
    ax2.text(i, v + max(memory_mb)*0.02, f"{v:.1f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 5. Custom Data — Plug in Your Own Embeddings

TurboQuant works with any embedding model — just pass in your numpy arrays.

In [ ]:
# Example: index your own embeddings
# Replace with your actual data:
#   vectors = np.load("my_embeddings.npy")
#   Or use sentence-transformers:
#   from sentence_transformers import SentenceTransformer
#   model = SentenceTransformer("all-MiniLM-L6-v2")
#   vectors = model.encode(your_texts)

# Demo with random data of different dimensions
for dim in [128, 384, 768, 1536]:
    vectors = np.random.randn(10000, dim).astype(np.float32)
    
    index = TurboQuantSearchIndex(dim=dim, bits=3)
    index.add(vectors)
    
    stats = index.stats()
    print(f"dim={dim:>5}: {stats['compression_ratio']} compression, "
          f"{stats['memory_mb']:.1f} MB (vs {vectors.nbytes / 1e6:.1f} MB uncompressed)")